In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# --- 1. SETUP & CONFIGURATION ---
model = YOLO("yolov8n.pt")
roi_points = []
px_to_meter = 0.04  
LOW_THRESH, HIGH_THRESH = 0.1, 0.2

# --- 2. LIVE ROI SELECTION LOGIC ---
def mouse_callback(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        roi_points.append((x, y))

cap = cv2.VideoCapture(0)
if not cap.isOpened():
    print("Error: Could not access webcam.")
    exit()

cv2.namedWindow("Draw ROI - Press ENTER when done")
cv2.setMouseCallback("Draw ROI - Press ENTER when done", mouse_callback)

print("Select ROI points on the live video. Press ENTER when finished.")

while True:
    ret, frame = cap.read()
    if not ret: break
    
    setup_frame = frame.copy()
    
    # Draw current points and lines on the live feed
    for i, pt in enumerate(roi_points):
        cv2.circle(setup_frame, pt, 5, (0, 0, 255), -1)
        if i > 0:
            cv2.line(setup_frame, roi_points[i-1], roi_points[i], (255, 255, 0), 2)
    
    cv2.imshow("Draw ROI - Press ENTER when done", setup_frame)
    
    # Press Enter (13) to finish, or 'q' to exit
    key = cv2.waitKey(1) & 0xFF
    if key == 13 and len(roi_points) >= 3: 
        break
    elif key == ord('q'):
        cap.release()
        cv2.destroyAllWindows()
        exit()

cv2.destroyWindow("Draw ROI - Press ENTER when done")

# Calculations
roi_np = np.array(roi_points, np.int32)
x_pts, y_pts = roi_np[:, 0], roi_np[:, 1]
area_px = 0.5 * np.abs(np.dot(x_pts, np.roll(y_pts, 1)) - np.dot(y_pts, np.roll(x_pts, 1)))
area_m2 = area_px * (px_to_meter**2)

# --- 3. REINFORCEMENT LEARNING SETUP ---
class CrowdEnv(gym.Env):
    def __init__(self):
        super(CrowdEnv, self).__init__()
        self.action_space = spaces.Discrete(3) 
        self.observation_space = spaces.Box(low=0, high=20, shape=(1,), dtype=np.float32)
        self.state = np.array([0], dtype=np.float32)

    def step(self, action, current_density):
        self.state = np.array([current_density], dtype=np.float32)
        reward = 0
        if action == 0 and current_density > HIGH_THRESH: reward -= 10
        if action == 2 and current_density < LOW_THRESH: reward -= 2
        if action == 2 and current_density > HIGH_THRESH: reward += 5
        return self.state, reward

# Initialize RL
rl_helper = CrowdEnv()
rl_model = PPO("MlpPolicy", rl_helper, verbose=0)
current_obs = np.array([0], dtype=np.float32)

# --- 4. MAIN PROCESSING LOOP ---
while True:
    # RL Decide Delay
    action, _ = rl_model.predict(current_obs)
    intervals = [1000, 200, 1] # ms
    rl_delay = intervals[action]
    
    ret, frame = cap.read()
    if not ret: break
    
    results = model(frame, verbose=False)
    person_count = 0
    overlay = frame.copy()
    
    for r in results:
        for box in r.boxes:
            if model.names[int(box.cls[0])] == 'person':
                coords = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, coords)
                feet = (int((x1 + x2) / 2), y2)
                
                if cv2.pointPolygonTest(roi_np, feet, False) >= 0:
                    person_count += 1
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                else:
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 150), 1)

    density = person_count / area_m2 if area_m2 > 0 else 0
    current_obs = np.array([density], dtype=np.float32)

    # UI Logic
    if density == 0: status, color = "EMPTY", (150, 150, 150)
    elif density < LOW_THRESH: status, color = "LOW", (0, 255, 0)
    elif density < HIGH_THRESH: status, color = "NORMAL", (0, 255, 255)
    else: status, color = "CROWDED", (0, 0, 255)

    cv2.fillPoly(overlay, [roi_np], (255, 255, 0))
    frame = cv2.addWeighted(overlay, 0.3, frame, 0.7, 0)
    cv2.polylines(frame, [roi_np], True, (255, 255, 0), 2)
    
    cv2.rectangle(frame, (0, 0), (frame.shape[1], 60), (30, 30, 30), -1)
    cv2.putText(frame, f"STATUS: {status} | RL MODE: {action}", (15, 25), 1, 1.5, color, 2)
    cv2.putText(frame, f"Count: {person_count} | Density: {density:.2f} p/m2", (15, 50), 1, 1, (255, 255, 255), 1)

    cv2.imshow("Crowd Intelligence App", frame)
    
    if cv2.waitKey(rl_delay) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Select ROI points on the live video. Press ENTER when finished.
